In [ ]:
import open3d as o3d
import os
import numpy as np
from tqdm.notebook import tqdm
import math
import json
import random
import uuid
import ifcopenshell
from ifcopenshell import template
import ifcopenshell.geom

#from src.visualisation import *
from src.ifc import *
from src.elements import create_pipe, create_elbow
from src.dataset import *
from src.preparation import *
from src.cloud import add_noise

# create_guid = lambda: ifcopenshell.guid.compress(uuid.uuid1().hex)
from numpy.random import default_rng

In [ ]:
random.seed(10)
rng = default_rng()

### CLOI Dataset Creation

The following section converts CLOI scans into a pcd dataset.

In [ ]:
# combine clouds
data_path = "/mnt/f/datasets/export/export/"
max_points = 4096

In [ ]:
classes = os.listdir(data_path)
print(classes)

In [ ]:
all_classes = []
element_count = 0
error_count = 0
for i, cl in enumerate(classes):
    all_elements = []
    class_path = data_path + cl
    elements = os.listdir(class_path)
    for j, el in tqdm(enumerate(elements)):
        try:
            element = np.loadtxt(class_path + "/" + el)

            # downsample
            if len(element) > 0 and element.ndim == 2 and element.shape[1] == 4:
                if len(element) > max_points:
                    # idx = np.random.randint(element.shape[0], size=max_points)
                    # element = element[idx :]
                    element = np.random.permutation(element)[:max_points]

                element = np.delete(element, 3, axis=1)  # remove point index
                element = np.insert(
                    element, 3, values=[element_count], axis=1
                )  # add element index
                # print(element.shape)
                element_count += 1
                all_elements.append(element)
        except Exception as E:
            error_count += 1

    all_elements = np.vstack(all_elements)
    all_elements = np.insert(all_elements, 4, values=[i], axis=1)  # add class index
    all_classes.append(all_elements)

all_classes = np.concatenate(all_classes)
print(all_classes.shape)
print("errors: ", error_count)

# print(points[0])

In [ ]:
pcd = o3d.t.geometry.PointCloud()
points = all_classes[:, 0:3]
el_index = [[i] for i in all_classes[:, 3]]
cl_index = [[i] for i in all_classes[:, 4]]


pcd.point["positions"] = o3d.core.Tensor(points)
pcd.point["elements"] = o3d.core.Tensor(el_index)
pcd.point["classes"] = o3d.core.Tensor(cl_index)

o3d.t.io.write_point_cloud("water2.pcd", pcd)

## Pipe parameter detection

### Generation of synethetic IFC element dataset

#### Dataset creation process
1. Generate params for element model
2. Generate ifc models
3. Convert to obj models using ifcConvert (ifc2obj.py script (synthetic) OR element_to_obj function (BP)) 
4. Convert to partially occluded EXR images using render_depth.py script *./blender -b -P render_depth.py ../industrial-facility-relationships/output/obj/ output/*
5. Convert to point clouds using process_exr.py script *python process_exr.py output/exr/ output/intrinsics.txt output/*
6. Combine multiple views of object to create training and testing datasets (for this, the metadata generated in step 3 must be pasted to individual metadata files for each class. This steps outputs a metadata_new file. set multiple=False for BP datasets)

BP datasets are created by following steps 3->6 above after splitting extracted IFC into multiple IFCs (to fix overlapping pieces)

##### Step 1 & 2. IFC model generation


In [ ]:
density = 2048
sample_size = 4096
config_path = "config/pipeline.json"
pcd_path = "/home/haritha/documents/blender-2.79-linux-glibc219-x86_64/output/pcd/"
blueprint = "data/sample.ifc"
num_scans = 16

**Elbow - modelled as an IfcRevolvedAreaSolid model**

*params:*

- position - 3D coordinate
- direction - 3D vector, axis of extrusion (normal to axis of revolution) (z>=0)
- axis_position - 2D coordinate, relative to position
- angle - angle of revolution (0 -> pi)
- radius


**Pipe - modelled as an IfcExtrudedAreaSolid model**

*params:*
    
- position - 3D coordinate
- extrusion_direction - 3D vector (z>=0)
- length
- radius


**Tee - modelled as a combination of 2 IfcRevolvedAreaSolid models**

The two pipes are each susbtracted from the other to create an IfcCsgSolid using an IfcBooleanResult.

*params:*
    
- position - 3D coordinate
- extrusion_direction1 - 3D vector (z>=0)
- extrusion_direction2 - 3D vector (z>=0)
- length1
- length2 - percentage of length1
- tee angle - 90 degrees or within an angle range
- radius1
- radius2 - same as radius1 or percentage of radius1


In [ ]:
synthetic_dataset(config_path, sample_size, "lbeam", 'industrial_completion', blueprint, 0)

*Use external scripts to convert above IFC dataset into ocluded PCD dataset. (step 3, 4 & 5)*

##### Step 6. Test / train dataset creation

1. merge multiple views
2. sample to standard density
3. generate training and testing dataset



In [ ]:
create_merged_dataset(
    pcd_path, "occluded_ood/", "ibeam", num_scans, density, 2, 0.1, False, multiple=True
)

In [ ]:
# create occluded dataset with non occluded ground truth for point cloud completion
pcd_path = "industrial_completion/lbeam/output/pcd/"

create_completion_dataset(
    pcd_path, "industrial_completion/", "lbeam", num_scans, density, 1, .1, False, multiple=False, noise=True
)

In [ ]:
# create noisy dataset with non noisy ground truth for point cloud completion
pcd_path = "mesh_dataset/elbow/blender/pcd/"

create_denoising_dataset(
    pcd_path, "noise_dataset/", "elbow", num_scans, density, .2, False
)

In [ ]:
print(7 * sum([i for i in range(1, 5 + 1)]))

In [ ]:
# add noise  to existing dataset
input_dir = "output/tee/train/"
output_dir = "output/noisy/tee/train/"
noise_size = 128
cloud = o3d.geometry.PointCloud()

files = os.listdir(input_dir)

for f in tqdm(files):
    points = np.array(o3d.io.read_point_cloud(input_dir + f).points)
    noisy_points = add_noise(points, noise_size, rng)
    noisy_points = o3d.utility.Vector3dVector(noisy_points)
    cloud.points = noisy_points
    o3d.io.write_point_cloud(output_dir + f, cloud)

In [ ]:
# script to recover pipe metadata from metadata_new since the metadata file has magically gotten corrupted

new_m = "output/pipe/metadata_new.json"
f = open(new_m, "r")
metadata = json.load(f)
meta_dict = {}

for m in metadata:
    meta_dict[metadata[m]["initial_ifc"]] = {
        "radius": metadata[m]["radius"],
        "direction": metadata[m]["direction"],
        "length": metadata[m]["length"],
        "position": metadata[m]["position"],
    }

In [ ]:
print(meta_dict.keys())

In [ ]:
# move occlusion dataset into pcn file structure for point cloud completion

import os
import shutil
from pathlib import Path

def organize_point_clouds_for_completion_or_denoising(source_dir, new_root, element_class):
    """
    Organize point cloud files into partial and complete folders.
    
    Args:
        source_dir: Directory containing the .pcd files (0.pcd, 0_gt.pcd, 1.pcd, 1_gt.pcd, etc.)
        new_root: Root directory where new folder structure will be created
    """
    
    # Create root directories
    partial_dir = os.path.join(new_root, 'partial/' + element_class)
    complete_dir = os.path.join(new_root, 'complete/' + element_class)
    
    os.makedirs(partial_dir, exist_ok=True)
    os.makedirs(complete_dir, exist_ok=True)
    
    # Get all .pcd files
    pcd_files = [f for f in os.listdir(source_dir) if f.endswith('.pcd')]
    
    # Extract base indices (0, 1, 2, etc.)
    indices = set()
    for f in pcd_files:
        if f.endswith('_gt.pcd'):
            idx = f.replace('_gt.pcd', '')
        elif f.endswith('_gt_l.pcd'):
            continue
        else:
            idx = f.replace('.pcd', '')

        indices.add(idx)
    
    # Process each index
    for idx in sorted(indices, key=lambda x: int(x)):
        partial_file = os.path.join(source_dir, f'{idx}.pcd')
        gt_file = os.path.join(source_dir, f'{idx}_gt.pcd')
        
        # Copy partial cloud
        if os.path.exists(partial_file):
            partial_idx_dir = os.path.join(partial_dir, idx)
            os.makedirs(partial_idx_dir, exist_ok=True)
            shutil.copy2(partial_file, os.path.join(partial_idx_dir, '00.pcd'))
            print(f"Copied {idx}.pcd -> {partial_idx_dir}/00.pcd")
        
        # Copy GT cloud (without _gt suffix)
        if os.path.exists(gt_file):
            shutil.copy2(gt_file, os.path.join(complete_dir, f'{idx}.pcd'))
            print(f"Copied {idx}_gt.pcd -> {complete_dir}/{idx}.pcd")

element_class = "lbeam"
dataset_root = "industrial_completion/"

source_directory = dataset_root + element_class + "/test/"
output_root = dataset_root + "test/"
organize_point_clouds_for_completion_or_denoising(source_directory, output_root, element_class)

source_directory = dataset_root + element_class + "/train/"
output_root = dataset_root + "train/"
organize_point_clouds_for_completion_or_denoising(source_directory, output_root, element_class)
print("Done!")

In [ ]:
def organize_point_clouds_for_reconstruction(source_dir, new_root, element_class):
    """
    Organize point cloud files into partial and complete folders for autoencoder reconstruction.
    Both input (partial) and output (complete) will use the ground truth point clouds.
    
    Args:
        source_dir: Directory containing the .pcd files (0.pcd, 0_gt.pcd, 1.pcd, 1_gt.pcd, etc.)
        new_root: Root directory where new folder structure will be created
        element_class: The element class string (e.g. 'ibeam', 'elbow')
    """
    
    # Create root directories
    partial_dir = os.path.join(new_root, 'partial/' + element_class)
    complete_dir = os.path.join(new_root, 'complete/' + element_class)
    
    os.makedirs(partial_dir, exist_ok=True)
    os.makedirs(complete_dir, exist_ok=True)
    
    # Get all .pcd files
    pcd_files = [f for f in os.listdir(source_dir) if f.endswith('.pcd')]
    
    # Extract base indices (0, 1, 2, etc.)
    indices = set()
    for f in pcd_files:
        if f.endswith('_gt.pcd'):
            idx = f.replace('_gt.pcd', '')
        elif f.endswith('_gt_l.pcd'):
            continue
        else:
            idx = f.replace('.pcd', '')

        indices.add(idx)
    
    # Process each index
    for idx in sorted(indices, key=lambda x: int(x)):
        # For reconstruction, both input and output are the GT
        gt_file = os.path.join(source_dir, f'{idx}_gt.pcd')
        
        # Copy GT cloud to the partial folder as the input (named 00.pcd)
        if os.path.exists(gt_file):
            partial_idx_dir = os.path.join(partial_dir, idx)
            os.makedirs(partial_idx_dir, exist_ok=True)
            shutil.copy2(gt_file, os.path.join(partial_idx_dir, '00.pcd'))
            print(f"Copied GT {idx}_gt.pcd -> {partial_idx_dir}/00.pcd (as partial/input)")
        
        # Copy GT cloud to the complete folder as the target (without _gt suffix)
        if os.path.exists(gt_file):
            shutil.copy2(gt_file, os.path.join(complete_dir, f'{idx}.pcd'))
            print(f"Copied GT {idx}_gt.pcd -> {complete_dir}/{idx}.pcd (as complete/target)")
            
element_class = "tee"
dataset_root = "autoencoder_dataset/"

source_directory = "noise_dataset/" + element_class + "/test/"
output_root = dataset_root + "test/"
organize_point_clouds_for_reconstruction(source_directory, output_root, element_class)

source_directory = "noise_dataset/" + element_class + "/train/"
output_root = dataset_root + "train/"
organize_point_clouds_for_reconstruction(source_directory, output_root, element_class)
print("Done!")

### create dataset to perform parametric modelling on BP helios dataset

In [ ]:
# convert completion dataset into param modelling dataset

import open3d as o3d
import numpy as np
import os
import json
import matplotlib.pyplot as plt
from glob import glob
from tqdm import tqdm

def normalize_pc(points):
    """
    Centres the point cloud at the origin and scales it such that 
    the mean of x^2 + y^2 + z^2 equals 1.
    """
    # 1. Centre the cloud
    centroid = np.mean(points, axis=0)
    points = points - centroid
    
    # 2. Scale so that average x^2 + y^2 + z^2 = 1
    norm_factor = np.sqrt(np.mean(np.sum(points**2, axis=1)))
    
    if norm_factor > 0:
        points = points / norm_factor
        
    return points, centroid, norm_factor

def resample_pc(points, target_num=2048):
    """
    Resamples the point cloud to exactly target_num points.
    Uses random choice with replacement for upsampling and without replacement for downsampling.
    """
    curr_num = points.shape[0]
    if curr_num >= target_num:
        indices = np.random.choice(curr_num, target_num, replace=False)
    else:
        indices = np.random.choice(curr_num, target_num, replace=True)
    return points[indices]

# Paths configured for your workspace
source_dir = '/home/haritha/documents/experiments/bp_completion/test/partial_flanges/02691156'
output_base_dir = '/home/haritha/documents/industrial-facility-relationships/data/helios/flange'
output_dir = os.path.join(output_base_dir, 'test')

os.makedirs(output_dir, exist_ok=True)

# Find all source 00.pcd files
pcd_files = glob(os.path.join(source_dir, '*', '00.pcd'))
print(f"Found {len(pcd_files)} files to process.")

# Dictionary to hold metadata for JSON
metadata = {}

# List to hold original point counts
input_sizes = []

for pcd_path in tqdm(pcd_files, desc="Processing"):
    # Use parent folder name as the original ID
    item_id_str = os.path.basename(os.path.dirname(pcd_path))
    
    try:
        pcd = o3d.io.read_point_cloud(pcd_path)
        points = np.asarray(pcd.points)
        
        num_points = len(points)
        if num_points == 0:
            continue
            
        # Record all valid file sizes for the histogram
        input_sizes.append(num_points)
        
        # Filter out point clouds with fewer than 100 points
        if num_points < 50:
            continue
            
        # Pre-processing steps
        points, centroid, norm_factor = normalize_pc(points)
        points = resample_pc(points, 2048)
        
        # Create new PointCloud object
        new_pcd = o3d.geometry.PointCloud()
        new_pcd.points = o3d.utility.Vector3dVector(points)
        
        # Save to new dataset structure
        save_path = os.path.join(output_dir, f"{item_id_str}.pcd")
        o3d.io.write_point_cloud(save_path, new_pcd)
        
        # Store metadata using the original ID
        item_id_int = int(item_id_str)
        metadata[item_id_str] = {
            "mean": centroid.tolist(),
            "norm_factor": float(norm_factor),
            "id": item_id_int
        }
        
    except Exception as e:
        print(f"Error processing {pcd_path}: {e}")

# Save the metadata dictionary to a JSON file
metadata_path = os.path.join(output_base_dir, 'metadata_new.json')
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=4)

# Plot and save histogram of input point cloud sizes
if input_sizes:
    plt.figure(figsize=(10, 6))
    plt.hist(input_sizes, bins=30, edgecolor='black')
    
    # Add a vertical line to indicate the filter cut-off
    plt.axvline(x=100, color='red', linestyle='dashed', linewidth=2, label='Filter Cut-off (<100)')
    plt.legend()
    
    plt.title('Distribution of Input Point Cloud Sizes')
    plt.xlabel('Number of Points')
    plt.ylabel('Frequency')
    
    plot_path = os.path.join(output_dir, 'input_sizes_histogram.png')
    plt.savefig(plot_path, bbox_inches='tight')
    plt.close()

print(f"\nDone! Dataset created at: {output_dir}")
print(f"Metadata saved at: {metadata_path}")
print(f"Histogram saved at: {plot_path}")

In [ ]:
from pathlib import Path

def filter_pcd_folder(folder_all_path, folder_subset_path):
    folder_all = Path(folder_all_path)
    folder_subset = Path(folder_subset_path)

    # Create a set of filenames present in the subset folder
    subset_filenames = {f.name for f in folder_subset.glob('*.pcd')}

    # Iterate through the main folder and delete files missing from the subset
    deleted_count = 0
    for pcd_file in folder_all.glob('*.pcd'):
        if pcd_file.name not in subset_filenames:
            pcd_file.unlink()
            deleted_count += 1
            
    print(f"Operation complete. Deleted {deleted_count} files from {folder_all.name}.")

filter_pcd_folder('/home/haritha/documents/experiments/PointAttN/completed_bp_clouds/flange/test', '/home/haritha/documents/industrial-facility-relationships/data/helios/flange/test')

In [ ]:
# merge pcds

import open3d as o3d
import numpy as np
from pathlib import Path

def merge_pcds_recursively(input_folder, output_filepath):
    # rglob finds all .pcd files in the folder and all its subdirectories
    pcd_files = list(Path(input_folder).rglob('*.pcd'))
    
    if not pcd_files:
        print("No .pcd files found in the specified directory.")
        return

    all_points = []
    
    for pcd_file in pcd_files:
        try:
            pcd = o3d.io.read_point_cloud(str(pcd_file))
            points = np.asarray(pcd.points)
            
            if len(points) > 0:
                all_points.append(points)
                
        except Exception as e:
            print(f"Error reading {pcd_file}: {e}")
            
    if not all_points:
        print("No valid points found to merge.")
        return

    # Concatenate all point arrays vertically
    merged_points = np.vstack(all_points)
    
    # Create the final merged point cloud
    merged_pcd = o3d.geometry.PointCloud()
    merged_pcd.points = o3d.utility.Vector3dVector(merged_points)
    
    # Save the result
    o3d.io.write_point_cloud(output_filepath, merged_pcd)
    
    print(f"Successfully merged {len(all_points)} files.")
    print(f"Total points in merged cloud: {len(merged_points)}")
    print(f"Saved to: {output_filepath}")

merge_pcds_recursively('/home/haritha/documents/experiments/bp_completion/test/partial_pipes', '/home/haritha/documents/experiments/bp_completion/test/output_merged_p_part.pcd')